<a href="https://colab.research.google.com/github/GuilhermeNunesP/Trabalhos-de-python/blob/main/Trabalho_Analise_de_dados_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.ticker import MaxNLocator
from plotly.subplots import make_subplots
from google.colab import files
from IPython.display import display , HTML
from io import BytesIO

In [7]:
arquivo = files.upload()
nome_arquivo = next(iter(arquivo))
df = pd.read_excel(nome_arquivo, sheet_name="Jogos")
df.columns = df.columns.str.strip()

colunas_numericas = [
    "ano_lancamento",
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

colunas_texto = [
    "nome_jogo",
    "genero",
    "genero_detalhado",
    "plataforma_original",
    "franquia_principal",
    "sub_franquia",
    "desenvolvedora_principal"
]

for coluna in colunas_texto:
    df[coluna] = df[coluna].astype("string").str.strip()
    df[coluna] = df[coluna].replace("", pd.NA)

dados = df.copy()
dados = dados[dados["nome_jogo"].notna()].copy()
print("\nBase preparada!")
print("Jogos disponíveis:", len(dados))
display(dados.head())

Saving Nintendo_Analise_de_Dados_v2.xlsx to Nintendo_Analise_de_Dados_v2 (1).xlsx

Base preparada!
Jogos disponíveis: 540


,id,nome_jogo,ano_lancamento,genero,genero_detalhado,plataforma_original,franquia_principal,sub_franquia,desenvolvedora_principal,sub_desenvolvedora,vendas_milhoes,nota_critica_100,nota_publico_100
0,J0024,Donkey Kong,1982,Ação,Ação e reflexos,Nintendo Game & Watch,Mario,Donkey Kong,Nintendo,NaN,NaN,NaN,NaN
1,J0036,Donkey Kong 3,1983,Ação,Tiro e arcade,Arcade,Mario,Donkey Kong,Nintendo,NaN,NaN,NaN,NaN
2,J0045,Donkey Kong 3,1984,Ação,Ação e reflexos,Nintendo Game & Watch,Mario,Donkey Kong,Nintendo,NaN,NaN,NaN,NaN
3,J0044,Donkey Kong Circus,1984,Ação,Ação e equilíbrio,Nintendo Game & Watch,Mario,Donkey Kong,Nintendo,NaN,NaN,NaN,NaN
4,J0035,Donkey Kong II,1983,Ação,Ação e reflexos,Nintendo Game & Watch,Mario,Donkey Kong,Nintendo,NaN,NaN,NaN,NaN


In [8]:
colunas_numericas = [
    "ano_lancamento",
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(
        dados[coluna], errors="coerce"
    )


dados.loc[
    dados["vendas_milhoes"] < 0,
    "vendas_milhoes"
] = np.nan


for coluna in ["nota_critica_100", "nota_publico_100"]:
    dados.loc[
        ~dados[coluna].between(0, 100),
        coluna
    ] = np.nan

total_jogos = dados["nome_jogo"].nunique()

vendas_totais = dados["vendas_milhoes"].sum(min_count=1)

media_critica = dados["nota_critica_100"].mean()

media_publico = dados["nota_publico_100"].mean()

base_franquias = dados.dropna(
    subset=["franquia_principal", "vendas_milhoes"]
).copy()

base_franquias["franquia_principal"] = (
    base_franquias["franquia_principal"]
    .astype("string")
    .str.strip()
)

base_franquias = base_franquias[
    base_franquias["franquia_principal"].ne("")
]

franquias = (
    base_franquias
    .groupby("franquia_principal")["vendas_milhoes"]
    .sum()
    .nlargest(15)
    .sort_values(ascending=True)
    .reset_index()
)

vendas_ano = (
    dados.dropna(
        subset=["ano_lancamento", "vendas_milhoes"]
    )
    .groupby("ano_lancamento", as_index=False)
    ["vendas_milhoes"]
    .sum()
    .sort_values("ano_lancamento")
)

vendas_ano["ano_lancamento"] = (
    vendas_ano["ano_lancamento"].astype(int)
)

base_consoles = dados.dropna(
    subset=["plataforma_original", "vendas_milhoes"]
).copy()

base_consoles["plataforma_original"] = (
    base_consoles["plataforma_original"]
    .astype("string")
    .str.strip()
)

base_consoles = base_consoles[
    base_consoles["plataforma_original"].ne("")
]

consoles = (
    base_consoles
    .groupby("plataforma_original")["vendas_milhoes"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

lancamentos_ano = (
    dados.dropna(subset=["ano_lancamento"])
    .groupby("ano_lancamento")
    .size()
    .reset_index(name="quantidade_jogos")
    .sort_values("ano_lancamento")
)

lancamentos_ano["ano_lancamento"] = (
    lancamentos_ano["ano_lancamento"].astype(int)
)

fundo = "#F2F4F8"
texto = "#253247"

cores_indicadores = [
    "#3978D4",
    "#169B86",
    "#8A5CC7",
    "#E39A35"
]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Franquias mais vendidas",
        "Evolução das vendas por ano",
        "Vendas acumuladas por console",
        "Quantidade de jogos lançados por ano"
    ),
    horizontal_spacing=0.12,
    vertical_spacing=0.19
)

fig.add_trace(
    go.Bar(
        x=franquias["vendas_milhoes"],
        y=franquias["franquia_principal"],
        orientation="h",
        marker_color="#3978D4",
        text=franquias["vendas_milhoes"].map(
            lambda x: (
                f"{x:,.1f} milhões"
                .replace(",", "X")
                .replace(".", ",")
                .replace("X", ".")
            )
        ),
        textposition="outside",
        cliponaxis=False,
        customdata=franquias["franquia_principal"],
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Vendas acumuladas: %{x:,.2f} milhões de copias"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=vendas_ano["ano_lancamento"],
        y=vendas_ano["vendas_milhoes"],
        mode="lines+markers",
        name="Vendas anuais",
        line=dict(color="#169B86", width=3),
        marker=dict(size=6),
        fill="tozeroy",
        fillcolor="rgba(22,155,134,0.12)",
        hovertemplate=(
            "<b>Ano: %{x}</b><br>"
            "Vendas: %{y:,.2f} milhões de copias"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=consoles["plataforma_original"],
        y=consoles["vendas_milhoes"],
        marker_color="#8A5CC7",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Vendas acumuladas: %{y:,.2f} milhões de copias"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Bar(
        x=lancamentos_ano["ano_lancamento"],
        y=lancamentos_ano["quantidade_jogos"],
        marker_color="#E39A35",
        hovertemplate=(
            "<b>Ano: %{x}</b><br>"
            "Jogos lançados: %{y}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=2
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#E0E5EC",
    zeroline=False
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#E0E5EC",
    zeroline=False
)

fig.update_xaxes(
    tickangle=-35,
    row=2,
    col=1
)

fig.update_xaxes(
    tickmode="linear",
    dtick=5,
    tickformat="d",
    tickangle=-45,
    row=1,
    col=2
)

fig.update_xaxes(
    tickmode="linear",
    dtick=5,
    tickformat="d",
    tickangle=-45,
    row=2,
    col=2
)

fig.update_layout(
    height=900,
    paper_bgcolor=fundo,
    plot_bgcolor="white",
    font=dict(
        family="Arial",
        color=texto,
        size=11
    ),
    title=dict(
        text="<b>Desempenho dos Jogos da Nintendo</b>",
        x=0.5,
        xanchor="center",
        font=dict(size=25, color=texto)
    ),
    showlegend=False,
    margin=dict(t=130, l=45, r=35, b=80),
    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Arial"
    ),
    bargap=0.25
)

fig.show()

def formatar_milhoes(valor):
    if pd.isna(valor):
        return "N/D"

    return (
        f"{valor:,.2f} mi"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def formatar_nota(valor):
    if pd.isna(valor):
        return "N/D"

    return f"{valor:.2f}/100".replace(".", ",")


def formatar_inteiro(valor):
    return f"{valor:,}".replace(",", ".")

cartoes = [
    {
        "titulo": "TOTAL DE JOGOS",
        "valor": formatar_inteiro(total_jogos),
        "cor": "#3978D4"
    },
    {
        "titulo": "VENDAS TOTAIS",
        "valor": formatar_milhoes(vendas_totais),
        "cor": "#169B86"
    },
    {
        "titulo": "MÉDIA DA CRÍTICA",
        "valor": formatar_nota(media_critica),
        "cor": "#8A5CC7"
    },
    {
        "titulo": "MÉDIA DO PÚBLICO",
        "valor": formatar_nota(media_publico),
        "cor": "#E39A35"
    }
]

html_cartoes = """
<div style="
    display:grid;
    grid-template-columns:repeat(4,minmax(0,1fr));
    gap:14px;
    font-family:Arial,sans-serif;
    margin:20px 0 28px 0;
">
"""

for cartao in cartoes:
    html_cartoes += f"""
    <div style="
        background:{cartao['cor']};
        color:white;
        padding:22px 14px;
        border-radius:12px;
        text-align:center;
        box-shadow:0 4px 10px rgba(0,0,0,0.10);
    ">
        <div style="font-size:13px;font-weight:bold;margin-bottom:12px;">
            {cartao['titulo']}
        </div>
        <div style="font-size:25px;font-weight:bold;">
            {cartao['valor']}
        </div>
    </div>
    """

html_cartoes += "</div>"

display(HTML(html_cartoes))




In [9]:
base = dados.copy()

colunas = [
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas:
    base[coluna] = pd.to_numeric(
        base[coluna],
        errors="coerce"
    )

base["nome_jogo"] = (
    base["nome_jogo"]
    .astype("string")
    .str.strip()
)

base = base[
    base["nome_jogo"].notna()
    & base["nome_jogo"].ne("")
].copy()

for coluna in ["nota_critica_100", "nota_publico_100"]:
    base.loc[
        ~base[coluna].between(0, 100),
        coluna
    ] = np.nan

top_vendas = (
    base.dropna(subset=["vendas_milhoes"])
    .query("vendas_milhoes >= 0")
    .sort_values("vendas_milhoes", ascending=False)
    .head(15)
    .sort_values("vendas_milhoes", ascending=True)
)

top_critica = (
    base.dropna(subset=["nota_critica_100"])
    .sort_values("nota_critica_100", ascending=False)
    .head(15)
    .sort_values("nota_critica_100", ascending=True)
)

top_publico = (
    base.dropna(subset=["nota_publico_100"])
    .sort_values("nota_publico_100", ascending=False)
    .head(15)
    .sort_values("nota_publico_100", ascending=True)
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_vendas["vendas_milhoes"],
        y=top_vendas["nome_jogo"],
        orientation="h",
        marker_color="#3978D4",
        customdata=top_vendas[["ano_lancamento"]]
        if "ano_lancamento" in top_vendas.columns
        else None,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Vendas: %{x:.2f} milhões"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
  updatemenus=[
   dict(
       type="dropdown",
       direction="down",
         x=0,
         xanchor="left",
         y=1.16,
         yanchor="top",
         buttons=[
            dict(
                label="Mais Vendidos",
                 method="update",
                 args=[
                        {
                            "x": [top_vendas["vendas_milhoes"]],
                            "y": [top_vendas["nome_jogo"]],
                            "marker.color": ["#3978D4"],
                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Vendas: %{x:.2f} milhões"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": "Jogos mais vendidos da Nintendo",
                            "xaxis": {
                                "title": "Vendas (milhões de unidades)"
                            },
                            "yaxis": {"title": ""}
                        }
                    ]
                ),
                dict(
                    label="Nota da Crítica",
                    method="update",
                    args=[
                        {
                            "x": [top_critica["nota_critica_100"]],
                            "y": [top_critica["nome_jogo"]],
                            "marker.color": ["#8A5CC7"],
                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Nota da crítica: %{x:.1f}/100"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": "Jogos mais bem avaliados pela crítica",
                            "xaxis": {"title": "Nota da crítica (0–100)"},
                            "yaxis": {"title": ""}
                        }
                    ]
                ),
                dict(
                    label="Nota do Público",
                    method="update",
                    args=[
                        {
                            "x": [top_publico["nota_publico_100"]],
                            "y": [top_publico["nome_jogo"]],
                            "marker.color": ["#169B86"],
                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Nota do público: %{x:.1f}/100"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": "Jogos mais bem avaliados pelo público",
                            "xaxis": {"title": "Nota do público (0–100)"},
                            "yaxis": {"title": ""}
                        }
                    ]
                )
            ]
        )
    ],
    title={
        "text": "Jogos mais vendidos da Nintendo",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="Vendas (milhões de unidades)",
    yaxis_title="",
    template="plotly_white",
    height=700,
    margin=dict(t=150, l=190, r=40, b=70),
    showlegend=False
)

fig.update_xaxes(showgrid=True, gridcolor="#E2E8F0")
fig.update_yaxes(showgrid=False, automargin=True)

fig.show()


In [ ]:
base = dados.copy()

colunas_numericas = [
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    base[coluna] = pd.to_numeric(
        base[coluna],
        errors="coerce"
    )

base["franquia_principal"] = (
    base["franquia_principal"]
    .astype("string")
    .str.strip()
)

base = base[
    base["franquia_principal"].notna()
    & base["franquia_principal"].ne("")
].copy()

base.loc[
    base["vendas_milhoes"] < 0,
    "vendas_milhoes"
] = np.nan

for coluna in ["nota_critica_100", "nota_publico_100"]:
    base.loc[
        ~base[coluna].between(0, 100),
        coluna
    ] = np.nan

estatisticas = (
    base.groupby("franquia_principal")
    .agg(
        vendas_totais=("vendas_milhoes", "sum"),
        jogos_com_vendas=("vendas_milhoes", "count"),

        media_critica=("nota_critica_100", "mean"),
        jogos_avaliados_critica=("nota_critica_100", "count"),

        media_publico=("nota_publico_100", "mean"),
        jogos_avaliados_publico=("nota_publico_100", "count"),

        jogos_totais=("nome_jogo", "nunique")
    )
    .reset_index()
)

top_vendas = (
    estatisticas[
        estatisticas["jogos_com_vendas"] > 0
    ]
    .sort_values(
        ["vendas_totais", "franquia_principal"],
        ascending=[False, True]
    )
    .head(15)
    .sort_values("vendas_totais", ascending=True)
)

top_critica = (
    estatisticas[
        estatisticas["jogos_avaliados_critica"] > 0
    ]
    .sort_values(
        ["media_critica", "franquia_principal"],
        ascending=[False, True]
    )
    .head(15)
    .sort_values("media_critica", ascending=True)
)

top_publico = (
    estatisticas[
        estatisticas["jogos_avaliados_publico"] > 0
    ]
    .sort_values(
        ["media_publico", "franquia_principal"],
        ascending=[False, True]
    )
    .head(15)
    .sort_values("media_publico", ascending=True)
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_vendas["vendas_totais"],
        y=top_vendas["franquia_principal"],
        orientation="h",
        marker_color="#3978D4",

        customdata=top_vendas[
            ["jogos_com_vendas", "jogos_totais"]
        ].to_numpy(),

        hovertemplate=(
            "<b>%{y}</b><br>"
            "Vendas acumuladas: %{x:.2f} milhões<br>"
            "Jogos com vendas registradas: %{customdata[0]}<br>"
            "Total de jogos da franquia: %{customdata[1]}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0,
            xanchor="left",
            y=1.18,
            yanchor="top",
            active=0,

            buttons=[
                dict(
                    label="Mais Vendidos",
                    method="update",
                    args=[
                        {
                            "x": [
                                top_vendas["vendas_totais"].tolist()
                            ],
                            "y": [
                                top_vendas["franquia_principal"].tolist()
                            ],
                            "marker.color": ["#3978D4"],

                            "customdata": [
                                top_vendas[
                                    ["jogos_com_vendas", "jogos_totais"]
                                ].to_numpy()
                            ],

                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Vendas acumuladas: %{x:.2f} milhões<br>"
                                "Jogos com vendas registradas: %{customdata[0]}<br>"
                                "Total de jogos da franquia: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Franquias mais vendidas da Nintendo",
                                "x": 0.5
                            },
                            "xaxis": {
                                "title": "Vendas acumuladas (milhões)",
                                "range": None
                            }
                        }
                    ]
                ),
                dict(
                    label="Nota da crítica",
                    method="update",
                    args=[
                        {
                            "x": [
                                top_critica["media_critica"].tolist()
                            ],
                            "y": [
                                top_critica["franquia_principal"].tolist()
                            ],
                            "marker.color": ["#8A5CC7"],

                            "customdata": [
                                top_critica[
                                    [
                                        "jogos_avaliados_critica",
                                        "jogos_totais"
                                    ]
                                ].to_numpy()
                            ],

                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Média da crítica: %{x:.2f}/100<br>"
                                "Jogos avaliados pela crítica: %{customdata[0]}<br>"
                                "Total de jogos da franquia: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Franquias mais bem avaliadas pela crítica",
                                "x": 0.5
                            },
                            "xaxis": {
                                "title": "Média da crítica (0–100)",
                                "range": [0, 100]
                            }
                        }
                    ]
                ),
                dict(
                    label="Nota do público",
                    method="update",
                    args=[
                        {
                            "x": [
                                top_publico["media_publico"].tolist()
                            ],
                            "y": [
                                top_publico["franquia_principal"].tolist()
                            ],
                            "marker.color": ["#169B86"],

                            "customdata": [
                                top_publico[
                                    [
                                        "jogos_avaliados_publico",
                                        "jogos_totais"
                                    ]
                                ].to_numpy()
                            ],

                            "hovertemplate": [
                                "<b>%{y}</b><br>"
                                "Média do público: %{x:.2f}/100<br>"
                                "Jogos avaliados pelo público: %{customdata[0]}<br>"
                                "Total de jogos da franquia: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Franquias mais bem avaliadas pelo público",
                                "x": 0.5
                            },
                            "xaxis": {
                                "title": "Média do público (0–100)",
                                "range": [0, 100]
                            }
                        }
                    ]
                )
            ]
        )
    ],

    title={
        "text": "Franquias mais vendidas da Nintendo",
        "x": 0.5,
        "xanchor": "center"
    },

    xaxis_title="Vendas acumuladas (milhões)",
    yaxis_title="",
    template="plotly_white",
    height=700,
    margin=dict(t=150, l=180, r=45, b=70),
    showlegend=False,
    font=dict(family="Arial", size=12, color="#253247")
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#E2E8F0",
    zeroline=False
)

fig.update_yaxes(
    showgrid=False,
    automargin=True
)

fig.show(config={"displayModeBar": False})

In [ ]:
base = dados.copy()

colunas_numericas = [
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    base[coluna] = pd.to_numeric(
        base[coluna],
        errors="coerce"
    )

base["plataforma_original"] = (
    base["plataforma_original"]
    .astype("string")
    .str.strip()
)

base = base[
    base["plataforma_original"].notna()
    & base["plataforma_original"].ne("")
].copy()


base.loc[
    base["vendas_milhoes"] < 0,
    "vendas_milhoes"
] = np.nan

for coluna in ["nota_critica_100", "nota_publico_100"]:
    base.loc[
        ~base[coluna].between(0, 100),
        coluna
    ] = np.nan



plataformas = (
    base.groupby("plataforma_original")
    .agg(
        vendas_totais=("vendas_milhoes", "sum"),
        jogos_com_vendas=("vendas_milhoes", "count"),
        media_critica=("nota_critica_100", "mean"),
        jogos_avaliados_critica=("nota_critica_100", "count"),
        media_publico=("nota_publico_100", "mean"),
        jogos_avaliados_publico=("nota_publico_100", "count"),
        quantidade_jogos=("nome_jogo", "count")
    )
    .reset_index()
)




dados_vendas = plataformas[
    plataformas["jogos_com_vendas"] > 0
].sort_values("vendas_totais", ascending=False)


dados_critica = plataformas[
    plataformas["jogos_avaliados_critica"] > 0
].sort_values("media_critica", ascending=False)


dados_publico = plataformas[
    plataformas["jogos_avaliados_publico"] > 0
].sort_values("media_publico", ascending=False)

def criar_bolhas(
    tabela,
    coluna_valor,
    coluna_quantidade,
    nome_metrica,
    cor,
    formato
):

    tamanhos = tabela["quantidade_jogos"].clip(lower=1)

    customdata = np.column_stack([
        tabela["quantidade_jogos"],
        tabela[coluna_quantidade]
    ])

    return go.Scatter(
        x=tabela["plataforma_original"],
        y=tabela[coluna_valor],
        mode="markers",
        name=nome_metrica,
        marker=dict(
            size=tamanhos,
            sizemode="area",
            sizeref=max(tamanhos.max(), 1) / 1800,
            sizemin=10,
            color=cor,
            opacity=0.78,
            line=dict(color="white", width=1.5)
        ),
        customdata=customdata,
        hovertemplate=(
            "<b>%{x}</b><br>"
            + nome_metrica + ": " + formato
            + "<br>Jogos cadastrados: %{customdata[0]}"
            + "<br>Jogos considerados na métrica: %{customdata[1]}"
            + "<extra></extra>"
        )
    )

fig = go.Figure()

fig.add_trace(
    criar_bolhas(
        dados_vendas,
        "vendas_totais",
        "jogos_com_vendas",
        "Vendas acumuladas (milhões)",
        "#3978D4",
        "%{y:,.2f} milhões de copias"
    )
)

fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0,
            xanchor="left",
            y=1.20,
            yanchor="top",
            active=0,
            buttons=[
                dict(
                    label="Vendas acumuladas",
                    method="update",
                    args=[
                        {
                            "x": [dados_vendas["plataforma_original"]],
                            "y": [dados_vendas["vendas_totais"]],
                            "marker.size": [
                                dados_vendas["quantidade_jogos"]
                            ],
                            "marker.color": ["#3978D4"],
                            "customdata": [
                                np.column_stack([
                                    dados_vendas["quantidade_jogos"],
                                    dados_vendas["jogos_com_vendas"]
                                ])
                            ],
                            "hovertemplate": [
                                "<b>%{x}</b><br>"
                                "Vendas acumuladas: %{y:,.2f} milhões"
                                "<br>Jogos cadastrados: %{customdata[0]}"
                                "<br>Jogos com vendas informadas: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Plataformas Nintendo com mais vendas acumuladas",
                                "x": 0.5
                            },
                            "yaxis": {
                                "title": "Vendas acumuladas (milhões)",
                                "rangemode": "tozero"
                            }
                        }
                    ]
                ),
                dict(
                    label="Nota da crítica",
                    method="update",
                    args=[
                        {
                            "x": [dados_critica["plataforma_original"]],
                            "y": [dados_critica["media_critica"]],
                            "marker.size": [
                                dados_critica["quantidade_jogos"]
                            ],
                            "marker.color": ["#8A5CC7"],
                            "customdata": [
                                np.column_stack([
                                    dados_critica["quantidade_jogos"],
                                    dados_critica["jogos_avaliados_critica"]
                                ])
                            ],
                            "hovertemplate": [
                                "<b>%{x}</b><br>"
                                "Média da crítica: %{y:.2f}/100"
                                "<br>Jogos cadastrados: %{customdata[0]}"
                                "<br>Jogos avaliados: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Plataformas Nintendo mais bem avaliados pela crítica",
                                "x": 0.5
                            },
                            "yaxis": {
                                "title": "Média da crítica (0–100)",
                                "range": [0, 100]
                            }
                        }
                    ]
                ),
                dict(
                    label="Nota do público",
                    method="update",
                    args=[
                        {
                            "x": [dados_publico["plataforma_original"]],
                            "y": [dados_publico["media_publico"]],
                            "marker.size": [
                                dados_publico["quantidade_jogos"]
                            ],
                            "marker.color": ["#169B86"],
                            "customdata": [
                                np.column_stack([
                                    dados_publico["quantidade_jogos"],
                                    dados_publico["jogos_avaliados_publico"]
                                ])
                            ],
                            "hovertemplate": [
                                "<b>%{x}</b><br>"
                                "Média do público: %{y:.2f}/100"
                                "<br>Jogos cadastrados: %{customdata[0]}"
                                "<br>Jogos avaliados: %{customdata[1]}"
                                "<extra></extra>"
                            ]
                        },
                        {
                            "title": {
                                "text": "Plataformas Nintendo mais bem avaliadas pelo público",
                                "x": 0.5
                            },
                            "yaxis": {
                                "title": "Média do público (0–100)",
                                "range": [0, 100]
                            }
                        }
                    ]
                )
            ]
        )
    ],
    title={
        "text": "Plataformas Nintendo com mais vendas acumuladas",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="Plataforma original",
    yaxis_title="Vendas acumuladas (milhões)",
    template="plotly_white",
    height=700,
    margin=dict(t=150, l=80, r=40, b=150),
    showlegend=False,
    font=dict(family="Arial", size=12, color="#253247")
)

fig.update_xaxes(
    tickangle=-45,
    showgrid=False,
    automargin=True
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#E2E8F0",
    zeroline=False
)

fig.show()


In [ ]:
base = dados.dropna(subset=["genero_detalhado"]).copy()

colunas_numericas = [
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    base[coluna] = pd.to_numeric(
        base[coluna], errors="coerce"
    )

base.loc[
    base["vendas_milhoes"] < 0,
    "vendas_milhoes"
] = float("nan")

for coluna in ["nota_critica_100", "nota_publico_100"]:
    base.loc[
        ~base[coluna].between(0, 100),
        coluna
    ] = float("nan")

categorias = base.groupby("genero_detalhado").agg(
    vendas_totais=("vendas_milhoes", "sum"),
    jogos_com_vendas=("vendas_milhoes", "count"),
    media_critica=("nota_critica_100", "mean"),
    media_publico=("nota_publico_100", "mean"),
    jogos=("nome_jogo", "count"),
    jogos_com_critica=("nota_critica_100", "count"),
    jogos_com_publico=("nota_publico_100", "count")
).reset_index()

categorias = categorias[
    categorias["media_critica"].notna()
    | categorias["media_publico"].notna()
    | (categorias["jogos_com_vendas"] > 0)
].copy()

categorias = (
    categorias.nlargest(15, "jogos")
    .sort_values("jogos", ascending=True)
)


fig = go.Figure()

fig.add_trace(go.Bar(
    y=categorias["genero_detalhado"],
    x=categorias["vendas_totais"],
    name="Vendas (milhões)",
    orientation="h",
    marker_color="#2CA25F",
    text=[
        f"{valor:.1f}" if pd.notna(valor) else "N/D"
        for valor in categorias["vendas_totais"]
    ],
    textposition="outside",
    cliponaxis=False,
    customdata=categorias[
        ["jogos", "jogos_com_vendas"]
    ].to_numpy(),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Vendas totais: %{x:.2f} milhões<br>"
        "Jogos na categoria: %{customdata[0]}<br>"
        "Jogos com vendas registradas: %{customdata[1]}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Bar(
    y=categorias["genero_detalhado"],
    x=categorias["media_critica"],
    name="Crítica (0–100)",
    orientation="h",
    marker_color="#2878B5",
    text=[
        f"{valor:.1f}" if pd.notna(valor) else "N/D"
        for valor in categorias["media_critica"]
    ],
    textposition="outside",
    cliponaxis=False,
    customdata=categorias[
        ["jogos", "jogos_com_critica"]
    ].to_numpy(),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Nota média da crítica: %{x:.2f}<br>"
        "Jogos na categoria: %{customdata[0]}<br>"
        "Jogos com nota da crítica: %{customdata[1]}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Bar(
    y=categorias["genero_detalhado"],
    x=categorias["media_publico"],
    name="Público (0–100)",
    orientation="h",
    marker_color="#F28E2B",
    text=[
        f"{valor:.1f}" if pd.notna(valor) else "N/D"
        for valor in categorias["media_publico"]
    ],
    textposition="outside",
    cliponaxis=False,
    customdata=categorias[
        ["jogos", "jogos_com_publico"]
    ].to_numpy(),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Nota média do público: %{x:.2f}<br>"
        "Jogos na categoria: %{customdata[0]}<br>"
        "Jogos com nota do público: %{customdata[1]}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Recepção por Gênero",
    template="plotly_white",
    barmode="group",
    height=800,

    xaxis=dict(
        title="Vendas (milhões) / Notas médias (0–100)",
        showgrid=True,
        gridcolor="lightgray",
        rangemode="tozero"
    ),

    yaxis=dict(
        title="Gêneros",
        automargin=True
    ),

    legend=dict(
        orientation="h",
        y=1.08,
        x=0
    ),

    margin=dict(
        l=180,
        r=100,
        t=130,
        b=70
    ),

    uniformtext_minsize=9,
    uniformtext_mode="hide"
)

fig.show(config={"displayModeBar": False})


categorias["diferenca_critica_publico"] = (
    categorias["media_critica"]
    - categorias["media_publico"]
)



In [ ]:
dados = df.copy()

colunas_numericas = [
    "vendas_milhoes",
    "nota_critica_100",
    "nota_publico_100"
]

for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(
        dados[coluna], errors="coerce"
    )

dados.loc[
    dados["vendas_milhoes"] < 0,
    "vendas_milhoes"
] = float("nan")

for coluna in ["nota_critica_100", "nota_publico_100"]:
    dados.loc[
        ~dados[coluna].between(0, 100),
        coluna
    ] = float("nan")

partes = []

for coluna in ["desenvolvedora_principal", "sub_desenvolvedora"]:

    parte = dados[
        [
            "nome_jogo",
            "vendas_milhoes",
            "nota_critica_100",
            "nota_publico_100",
            coluna
        ]
    ].copy()

    parte = parte.rename(
        columns={coluna: "desenvolvedora"}
    )

    parte["desenvolvedora"] = (
        parte["desenvolvedora"]
        .astype("string")
        .str.strip()
    )

    parte = parte[
        parte["desenvolvedora"].notna()
        & parte["desenvolvedora"].ne("")
        & parte["desenvolvedora"].str.lower().ne("n/d")
    ]

    partes.append(parte)

dados_dev = pd.concat(partes, ignore_index=True)

dados_dev = dados_dev.drop_duplicates(
    subset=["nome_jogo", "desenvolvedora"]
)


resumo_dev = dados_dev.groupby("desenvolvedora").agg(
    vendas_totais=("vendas_milhoes", "sum"),
    jogos_com_vendas=("vendas_milhoes", "count"),

    media_critica=("nota_critica_100", "mean"),
    jogos_avaliados_critica=("nota_critica_100", "count"),

    media_publico=("nota_publico_100", "mean"),
    jogos_avaliados_publico=("nota_publico_100", "count"),

    quantidade_jogos=("nome_jogo", "nunique")
).reset_index()

categorias = (
    resumo_dev[
        resumo_dev["jogos_com_vendas"] > 0
    ]
    .nlargest(15, "vendas_totais")
    .sort_values("vendas_totais", ascending=True)
)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=categorias["desenvolvedora"],
    x=categorias["vendas_totais"],
    name="Vendas (milhões)",
    orientation="h",
    marker_color="#2CA25F",

    text=[
        f"{valor:.1f}"
        for valor in categorias["vendas_totais"]
    ],
    textposition="outside",
    cliponaxis=False,

    customdata=categorias[
        ["jogos_com_vendas", "quantidade_jogos"]
    ].to_numpy(),

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Vendas acumuladas: %{x:.2f} milhões<br>"
        "Jogos usados no cálculo das vendas: %{customdata[0]}<br>"
        "Jogos associados à desenvolvedora: %{customdata[1]}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Bar(
    y=categorias["desenvolvedora"],
    x=categorias["media_critica"],
    name="Crítica (0–100)",
    orientation="h",
    marker_color="#2878B5",

    text=[
        f"{valor:.1f}" if pd.notna(valor) else "N/D"
        for valor in categorias["media_critica"]
    ],
    textposition="outside",
    cliponaxis=False,

    customdata=categorias[
        ["jogos_avaliados_critica"]
    ].to_numpy(),

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Nota média da crítica: %{x:.2f}/100<br>"
        "Jogos usados no cálculo da média: %{customdata[0]}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Bar(
    y=categorias["desenvolvedora"],
    x=categorias["media_publico"],
    name="Público (0–100)",
    orientation="h",
    marker_color="#F28E2B",

    text=[
        f"{valor:.1f}" if pd.notna(valor) else "N/D"
        for valor in categorias["media_publico"]
    ],
    textposition="outside",
    cliponaxis=False,

    customdata=categorias[
        ["jogos_avaliados_publico"]
    ].to_numpy(),

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Nota média do público: %{x:.2f}/100<br>"
        "Jogos usados no cálculo da média: %{customdata[0]}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Recepção das desenvolvedoras da Nintendo",
    template="plotly_white",
    barmode="group",
    height=800,

    xaxis=dict(
        title="Vendas (milhões) / Notas médias (0–100)",
        showgrid=True,
        gridcolor="lightgray",
        rangemode="tozero"
    ),

    yaxis=dict(
        title="Desenvolvedora",
        automargin=True
    ),

    legend=dict(
        orientation="h",
        y=1.08,
        x=0
    ),

    margin=dict(
        l=200,
        r=110,
        t=130,
        b=70
    ),

    uniformtext_minsize=9,
    uniformtext_mode="hide"
)

fig.show(config={"displayModeBar": False})